In [6]:
from dataclasses import dataclass
from pathlib import Path
import os
import shutil
from ultralytics import YOLO
from src.TALOS import logger


@dataclass(frozen=True)

class ModelTrainerConfig:
  root_dir: Path
  data_path: Path
  model_name: str
  epochs: int
  imgsz: int
  rect: bool
  batch: int
  device: str
  plots: bool

In [3]:
ls

data_ingestion.ipynb
data_transformation.ipynb
data_validation.ipynb
kalman_filter_and_hungarain_assignment.ipynb
model_train.ipynb
trials.ipynb


In [4]:
cd ..

/Users/gojuruakshith/Tactical-Aerospace-Localization-Objective-Scoring-TALOS-


In [5]:
from src.TALOS.config.configuration import DataTransformationConfig
from src.TALOS.constants import *
from src.TALOS.utils.common import read_yaml,create_directories

class ConfigurationManager:
    def __init__(self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])

    def get_model_train_config(self)->ModelTrainerConfig:
        config = self.config.model_train
        create_directories([config.root_dir])

        return ModelTrainerConfig(
            root_dir = config.root_dir,
            data_path=config.data_path,
            model_name = config.model_name,
            epochs = config.epochs,
            imgsz =  config.imgsz,
            rect =  config.rect,
            batch = config.batch,
            device = config.device,
            plots = config.plots
        )


In [ ]:
class ModelTrain:
    def __init__(self,config: ModelTrainerConfig):
        self.config = config
    def model_train(self):
        model = YOLO('yolo26s.pt')
        model.train(
        data=self.config.data_path,
        epochs=self.config.epochs,
        imgsz=self.config.imgsz,
        rect=self.config.rect,
        batch=self.config.batch,
        device=self.config.device,
        plots=self.config.plots
        )
    def move_trained_weights(self):
        try:
            source_path = Path("runs/detect/train3/weights/best.pt")
            target_dir = Path(self.config.root_dir)
            os.makedirs(target_dir, exist_ok=True)
            target_path = target_dir / "best.pt"
            if source_path.exists():
                shutil.move(str(source_path), str(target_path))
                logger.info(f"Successfully moved weights to: {target_path}")
                print(f"DONE: Model saved at {target_path}")
            else:
                logger.warning("best.pt not found! Check if training finished correctly.")
        except Exception as e:
            logger.error(f"Failed to move weights: {e}")

In [ ]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_train_config()
    model_trainer = ModelTrain(config = model_trainer_config)
    model_trainer.model_train()
    model_trainer.move_trained_weights()
except Exception as e:
    raise e